[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/52_opd_loss.ipynb)

# 🔴 Hard: OPD (On-Policy Distillation) Loss

*RLHF & Preference Losses*
Implement the **on-policy distillation** loss: match a student's next-token
distribution to one or more teachers, on sequences the *student* generated.

$$\mathcal{L} = T^2 \cdot \frac{\sum_t m_t \sum_k w_k\,
\mathbb{D}_{KL}\!\left[\pi_S \,\|\, \pi_{T_k}\right]_t}{\sum_t m_t}$$

with the **reverse** KL

$$\mathbb{D}_{KL}[\pi_S\|\pi_T] = \sum_v \pi_S(v)\big(\log \pi_S(v) - \log \pi_T(v)\big)$$

and both distributions taken at temperature $T$, i.e.
$\pi = \text{softmax}(\text{logits}/T)$.

### Signature
```python
def opd_loss(student_logits, teacher_logits, temperature=1.0,
             teacher_weights=None, mask=None):
    # student_logits: (batch, seq, vocab)
    # teacher_logits: (batch, seq, vocab) or (n_teachers, batch, seq, vocab)
    # teacher_weights: (n_teachers,) or None -> uniform
    # mask: (batch, seq) of 1.0 real / 0.0 padding, or None
    ...  # -> scalar
```

### Rules
- **Reverse** KL: the expectation is under the student, not the teacher
- Apply the temperature to **both** logit sets before softmax
- Multiply the result by $T^2$
- Support a single teacher `(B, S, V)` *and* a stack `(K, B, S, V)`
- `teacher_weights` defaults to uniform and should be normalised to sum to 1
- Mask-average over real tokens
- Use `jax.nn.log_softmax`; do not write `log(softmax(x))`

### Why the $T^2$ factor
Raising the temperature flattens both distributions, which shrinks the gradients
roughly as $1/T^2$. Multiplying the loss by $T^2$ cancels that, so the same
learning rate works across temperatures. Hinton's distillation paper introduces
this exact correction; forgetting it means every temperature change silently
becomes a learning-rate change.

### Forward vs reverse KL — the part that matters
| | Direction | Behaviour |
|---|---|---|
| Forward $\mathbb{D}[\pi_T\|\pi_S]$ | expectation under teacher | **mode-covering** — the student must put mass everywhere the teacher does, so it smears over all modes |
| Reverse $\mathbb{D}[\pi_S\|\pi_T]$ | expectation under student | **mode-seeking** — the student is only penalised where *it* puts mass, so it can safely ignore modes and commit to one |

Reverse KL is the right choice here: a smeared-out student that hedges across
every plausible continuation generates worse text than one that commits. The
cost is that the student can quietly drop teacher behaviours entirely.

### Why *on-policy* is the other half of the idea
Off-policy distillation trains on the teacher's own outputs, so the student only
ever sees states a competent model reaches. At inference the student drifts into
states its teacher never visited and has no idea what to do — the classic
exposure-bias / compounding-error failure. On-policy distillation samples from
the **student**, so the teacher supervises exactly the states the student
actually lands in. That is why the sequences here are the student's, and why the
loss is evaluated token-wise over them.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def opd_loss(student_logits, teacher_logits, temperature=1.0,
             teacher_weights=None, mask=None):
    """On-policy distillation loss (reverse KL, temperature-scaled).

    Args:
        student_logits:  (batch, seq, vocab)
        teacher_logits:  (batch, seq, vocab) or (n_teachers, batch, seq, vocab)
        temperature:     softmax temperature applied to both
        teacher_weights: (n_teachers,) mixing weights, or None for uniform
        mask:            (batch, seq) 1.0 real / 0.0 padding, or None

    Returns:
        Scalar loss.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

logits = jax.random.normal(jax.random.key(0), (2, 3, 6))

print("student == teacher:", float(opd_loss(logits, logits)))          # ~0
print("different teacher: ", float(opd_loss(logits, logits * 2.0)))    # > 0

# T^2 scaling keeps the loss on a comparable scale as temperature changes.
for T in (1.0, 2.0, 4.0):
    print(f"  T={T}: {float(opd_loss(logits, logits * 2.0, temperature=T)):.4f}")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("opd_loss")

# hint("opd_loss")      # stuck? nudge without the answer
# solution("opd_loss")  # spoiler: the reference implementation